# 📊 C3-Pipeline: Synthetic Data Generation

## Overview
This notebook generates synthetic hospital data simulating:
1. **EHR System** (Epic Clarity/Caboodle): `dim_encounters`, `fact_consult_orders`, `fact_consult_completions`
2. **Secure Messaging Platform** (TigerConnect/Vocera): `fact_communication_logs`

### Data Scale
| Table | Target Rows |
|:---|:---|
| `dim_encounters` | ~10,000 |
| `fact_consult_orders` | ~30,000 |
| `fact_communication_logs` | ~100,000 |
| `fact_consult_completions` | ~25,500 |

### Intentional Bottlenecks
- **Cardiology & Nephrology** consults have systematically longer turnaround times
- **Weekend orders** (Fri 5PM – Mon 8AM) have longer completion times
- **Legacy Pager** has higher unread rates (~20%) vs Secure App Chat (~5%)
- **Night shift orders** (7PM–7AM) have additional delays

## Setup & Dependencies

In [ ]:
# Install dependencies (uncomment for Google Colab)
# !pip install faker pandas numpy

import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta
import random
import os
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
fake = Faker()
Faker.seed(SEED)

# Output paths
RAW_DIR = os.path.join('data', 'raw')
SAMPLE_DIR = os.path.join('data', 'sample')
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(SAMPLE_DIR, exist_ok=True)

print('✅ Dependencies loaded and output directories created.')

## Configuration Constants

In [ ]:
# ==================== SCALE PARAMETERS ====================
NUM_ENCOUNTERS = 10_000
NUM_UNIQUE_PATIENTS = 6_000
NUM_ATTENDING_PROVIDERS = 200
NUM_NURSES = 300
NUM_SPECIALISTS = 100

# Date range: Jan 2024 – Dec 2025
DATE_START = datetime(2024, 1, 1)
DATE_END = datetime(2025, 12, 31)

# ==================== CLINICAL PARAMETERS ====================
# Admitting units and their LOS characteristics
UNITS = {
    'ICU':           {'weight': 0.15, 'avg_los_days': 6.2, 'std_los_days': 3.0},
    'MedSurg East':  {'weight': 0.25, 'avg_los_days': 3.8, 'std_los_days': 2.0},
    'MedSurg West':  {'weight': 0.25, 'avg_los_days': 3.8, 'std_los_days': 2.0},
    'Telemetry':     {'weight': 0.20, 'avg_los_days': 4.5, 'std_los_days': 2.5},
    'Emergency Dept': {'weight': 0.15, 'avg_los_days': 1.2, 'std_los_days': 0.8},
}

# ICD-10 diagnosis codes
DIAGNOSES = [
    ('I50.9', 'Heart Failure, Unspecified'),
    ('J44.1', 'COPD with Acute Exacerbation'),
    ('N17.9', 'Acute Kidney Injury, Unspecified'),
    ('I21.9', 'Acute Myocardial Infarction, Unspecified'),
    ('J18.9', 'Pneumonia, Unspecified'),
    ('K92.0', 'Gastrointestinal Hemorrhage'),
    ('E11.65', 'Type 2 Diabetes with Hyperglycemia'),
    ('I63.9', 'Cerebral Infarction, Unspecified'),
    ('A41.9', 'Sepsis, Unspecified'),
    ('S72.001A', 'Hip Fracture'),
    ('G40.909', 'Epilepsy, Unspecified'),
    ('J96.01', 'Acute Respiratory Failure with Hypoxia'),
    ('I48.91', 'Atrial Fibrillation, Unspecified'),
    ('K85.9', 'Acute Pancreatitis, Unspecified'),
    ('N39.0', 'Urinary Tract Infection'),
]

# Discharge dispositions with probabilities
DISCHARGE_DISPOSITIONS = [
    ('Home', 0.55),
    ('Skilled Nursing Facility', 0.20),
    ('Rehab', 0.13),
    ('Expired', 0.05),
    ('Against Medical Advice', 0.02),
    ('Home with Home Health', 0.05),
]

# Target specialties for consults
TARGET_SPECIALTIES = [
    'Cardiology', 'Nephrology', 'Neurology', 'Pulmonology',
    'Infectious Disease', 'Gastroenterology', 'Orthopedics',
    'Oncology', 'Psychiatry', 'Palliative Care'
]

# Requesting provider specialties
REQUESTING_SPECIALTIES = [
    'Internal Medicine', 'Emergency Medicine', 'Hospitalist',
    'Family Medicine', 'General Surgery'
]

# Communication channels with response time profiles (in minutes)
CHANNELS = {
    'Secure App Chat':      {'weight': 0.35, 'avg_read_min': 8,  'std_read_min': 5,  'unread_rate': 0.05},
    'Vocera Badge Call':    {'weight': 0.20, 'avg_read_min': 3,  'std_read_min': 2,  'unread_rate': 0.08},
    'Legacy Pager':         {'weight': 0.30, 'avg_read_min': 25, 'std_read_min': 15, 'unread_rate': 0.20},
    'Phone Call to Office': {'weight': 0.15, 'avg_read_min': 40, 'std_read_min': 20, 'unread_rate': 0.15},
}

# Specialty-specific turnaround times (order to bedside, in hours)
SPECIALTY_TAT = {
    'Cardiology':          {'avg_hours': 6.5, 'std_hours': 3.0},
    'Nephrology':          {'avg_hours': 5.8, 'std_hours': 2.5},
    'Neurology':           {'avg_hours': 4.8, 'std_hours': 2.0},
    'Pulmonology':         {'avg_hours': 4.2, 'std_hours': 2.0},
    'Infectious Disease':  {'avg_hours': 3.5, 'std_hours': 1.5},
    'Gastroenterology':    {'avg_hours': 3.0, 'std_hours': 1.5},
    'Orthopedics':         {'avg_hours': 3.2, 'std_hours': 1.5},
    'Oncology':            {'avg_hours': 4.0, 'std_hours': 2.0},
    'Psychiatry':          {'avg_hours': 7.0, 'std_hours': 3.5},
    'Palliative Care':     {'avg_hours': 5.0, 'std_hours': 2.5},
}

# Message outcomes by channel type
MESSAGE_OUTCOMES = {
    'Secure App Chat':      {'Read & Acknowledged': 0.60, 'Read - No Response': 0.25, 'Unread': 0.05, 'Left Voicemail': 0.00, 'Answered Live': 0.10},
    'Vocera Badge Call':    {'Read & Acknowledged': 0.30, 'Read - No Response': 0.10, 'Unread': 0.08, 'Left Voicemail': 0.02, 'Answered Live': 0.50},
    'Legacy Pager':         {'Read & Acknowledged': 0.40, 'Read - No Response': 0.25, 'Unread': 0.20, 'Left Voicemail': 0.05, 'Answered Live': 0.10},
    'Phone Call to Office': {'Read & Acknowledged': 0.20, 'Read - No Response': 0.10, 'Unread': 0.15, 'Left Voicemail': 0.35, 'Answered Live': 0.20},
}

print('✅ Configuration constants defined.')

## Helper Functions

In [ ]:
def random_timestamp(start: datetime, end: datetime) -> datetime:
    """Generate a random timestamp between start and end."""
    delta = end - start
    random_seconds = random.randint(0, int(delta.total_seconds()))
    return start + timedelta(seconds=random_seconds)


def is_weekend_window(ts: datetime) -> bool:
    """Check if timestamp falls in the weekend window (Fri 5PM to Mon 8AM)."""
    dow = ts.weekday()  # 0=Monday, 6=Sunday
    hour = ts.hour
    if dow == 4 and hour >= 17:  # Friday after 5PM
        return True
    if dow == 5 or dow == 6:  # Saturday or Sunday
        return True
    if dow == 0 and hour < 8:  # Monday before 8AM
        return True
    return False


def is_night_shift(ts: datetime) -> bool:
    """Check if timestamp falls in night shift (7PM to 7AM)."""
    return ts.hour >= 19 or ts.hour < 7


def generate_id_pool(prefix: str, count: int, width: int = 5) -> list:
    """Generate a pool of IDs like 'DOC-00001', 'PAT-00001', etc."""
    return [f"{prefix}-{str(i).zfill(width)}" for i in range(1, count + 1)]


print('✅ Helper functions defined.')

---
## Table 1: `dim_encounters`

Each row represents one patient hospital stay (admission to discharge).
- **~10,000 rows** with ~6,000 unique patients (some readmissions)
- Admissions span Jan 2024 – Dec 2025
- Unit-specific LOS distributions

In [ ]:
print('Generating dim_encounters...')

# Pre-generate ID pools
patient_ids = generate_id_pool('PAT', NUM_UNIQUE_PATIENTS)
provider_ids = generate_id_pool('DOC', NUM_ATTENDING_PROVIDERS)

# Unit selection setup
unit_names = list(UNITS.keys())
unit_weights = [UNITS[u]['weight'] for u in unit_names]

# Diagnosis setup
diag_codes = [d[0] for d in DIAGNOSES]
diag_descs = [d[1] for d in DIAGNOSES]

# Discharge disposition setup
disp_names = [d[0] for d in DISCHARGE_DISPOSITIONS]
disp_weights = [d[1] for d in DISCHARGE_DISPOSITIONS]

encounters = []

for i in range(NUM_ENCOUNTERS):
    enc_id = f"ENC-{str(i + 1).zfill(6)}"
    pat_id = random.choice(patient_ids)
    
    # Patient demographics
    age = int(np.clip(np.random.normal(65, 15), 18, 95))
    sex = random.choice(['M', 'F'])
    
    # Admitting unit
    unit = random.choices(unit_names, weights=unit_weights, k=1)[0]
    
    # Admission timestamp
    admit_ts = random_timestamp(DATE_START, DATE_END)
    
    # Length of stay based on unit
    avg_los = UNITS[unit]['avg_los_days']
    std_los = UNITS[unit]['std_los_days']
    los_days = max(0.5, np.random.normal(avg_los, std_los))  # minimum half day
    
    # Discharge timestamp (~2% still admitted → NULL)
    discharge_ts = admit_ts + timedelta(days=los_days)
    if discharge_ts > datetime(2025, 12, 31, 23, 59, 59):
        discharge_ts = None  # still admitted
    
    # Discharge disposition
    if discharge_ts is not None:
        disposition = random.choices(disp_names, weights=disp_weights, k=1)[0]
    else:
        disposition = None
    
    # Diagnosis
    diag_idx = random.randint(0, len(DIAGNOSES) - 1)
    
    encounters.append({
        'encounter_id': enc_id,
        'patient_id': pat_id,
        'patient_age': age,
        'patient_sex': sex,
        'admit_timestamp': admit_ts.strftime('%Y-%m-%d %H:%M:%S'),
        'discharge_timestamp': discharge_ts.strftime('%Y-%m-%d %H:%M:%S') if discharge_ts else None,
        'discharge_disposition': disposition,
        'primary_diagnosis_code': diag_codes[diag_idx],
        'primary_diagnosis_desc': diag_descs[diag_idx],
        'admitting_unit': unit,
        'attending_provider_id': random.choice(provider_ids),
    })

df_encounters = pd.DataFrame(encounters)

print(f'✅ Generated {len(df_encounters):,} encounters')
print(f'   Unique patients: {df_encounters["patient_id"].nunique():,}')
print(f'   Null discharges (still admitted): {df_encounters["discharge_timestamp"].isna().sum()}')
print(f'   Unit distribution:')
print(df_encounters['admitting_unit'].value_counts().to_string())
print()
df_encounters.head(3)

---
## Table 2: `fact_consult_orders`

Each row is a physician requesting a specialist consultation.
- **~30,000 rows** (avg ~3 per encounter)
- ~60% of encounters get at least one consult
- Priority: 70% Routine, 25% Urgent, 5% STAT
- Status: 85% Completed, 10% Cancelled, 5% Pending

In [ ]:
print('Generating fact_consult_orders...')

# Determine which encounters get consults (~60%)
encounters_with_consults = df_encounters.sample(frac=0.60, random_state=SEED)

# Calculate how many consults per encounter to reach ~30,000 total
# 30,000 / (10,000 * 0.60) = 5 consults per encounter with consults
TARGET_CONSULTS = 30_000
avg_consults_per_enc = TARGET_CONSULTS / len(encounters_with_consults)

consult_orders = []
consult_counter = 0

for _, enc in encounters_with_consults.iterrows():
    # Number of consults for this encounter (Poisson-like distribution)
    n_consults = max(1, int(np.random.poisson(avg_consults_per_enc - 1) + 1))
    n_consults = min(n_consults, 10)  # cap at 10
    
    admit_ts = datetime.strptime(enc['admit_timestamp'], '%Y-%m-%d %H:%M:%S')
    
    if enc['discharge_timestamp'] is not None and pd.notna(enc['discharge_timestamp']):
        discharge_ts = datetime.strptime(enc['discharge_timestamp'], '%Y-%m-%d %H:%M:%S')
    else:
        discharge_ts = admit_ts + timedelta(days=7)  # assume 7 days for still-admitted
    
    # Ensure we have at least a few hours between admit and discharge for orders
    if (discharge_ts - admit_ts).total_seconds() < 7200:  # less than 2 hours
        discharge_ts = admit_ts + timedelta(hours=6)
    
    for j in range(n_consults):
        consult_counter += 1
        consult_id = f"CONS-{str(consult_counter).zfill(7)}"
        
        # Order timestamp: random time during the encounter
        order_ts = random_timestamp(admit_ts + timedelta(hours=1), discharge_ts - timedelta(hours=1))
        
        # Priority
        priority = random.choices(
            ['Routine', 'Urgent', 'STAT'],
            weights=[0.70, 0.25, 0.05],
            k=1
        )[0]
        
        # Order status
        status = random.choices(
            ['Completed', 'Cancelled', 'Pending'],
            weights=[0.85, 0.10, 0.05],
            k=1
        )[0]
        
        # Target specialty
        target_spec = random.choice(TARGET_SPECIALTIES)
        
        consult_orders.append({
            'consult_order_id': consult_id,
            'encounter_id': enc['encounter_id'],
            'requesting_provider_id': enc['attending_provider_id'],
            'requesting_provider_specialty': random.choice(REQUESTING_SPECIALTIES),
            'target_specialty': target_spec,
            'order_timestamp': order_ts.strftime('%Y-%m-%d %H:%M:%S'),
            'priority': priority,
            'order_status': status,
        })

df_consult_orders = pd.DataFrame(consult_orders)

print(f'✅ Generated {len(df_consult_orders):,} consult orders')
print(f'   Encounters with consults: {df_consult_orders["encounter_id"].nunique():,}')
print(f'   Priority distribution:')
print(df_consult_orders['priority'].value_counts(normalize=True).round(3).to_string())
print(f'   Status distribution:')
print(df_consult_orders['order_status'].value_counts(normalize=True).round(3).to_string())
print(f'   Specialty distribution:')
print(df_consult_orders['target_specialty'].value_counts().to_string())
print()
df_consult_orders.head(3)

---
## Table 3: `fact_communication_logs`

Each row is one communication attempt by a nurse to reach a specialist.
- **~100,000 rows** (avg ~3.3 messages per consult order)
- 4 channels with distinct response time profiles
- ~12% overall unread rate
- Channel-specific unread rates and outcomes

In [ ]:
print('Generating fact_communication_logs...')

# Pre-generate nurse/clerk IDs
nurse_ids = generate_id_pool('RN', NUM_NURSES)
specialist_ids = generate_id_pool('DOC', NUM_SPECIALISTS, width=4)

# Channel setup
channel_names = list(CHANNELS.keys())
channel_weights = [CHANNELS[c]['weight'] for c in channel_names]

comm_logs = []
msg_counter = 0

for _, consult in df_consult_orders.iterrows():
    # Number of communication attempts (1 to 8, weighted toward 2-4)
    n_messages = max(1, min(8, int(np.random.exponential(2.3) + 1)))
    
    order_ts = datetime.strptime(consult['order_timestamp'], '%Y-%m-%d %H:%M:%S')
    
    # Assign a specialist for this consult's messages
    recipient_id = random.choice(specialist_ids)
    
    # Assign a nurse for this consult
    sender_id = random.choice(nurse_ids)
    sender_role = random.choices(['Registered Nurse', 'Unit Clerk'], weights=[0.90, 0.10], k=1)[0]
    
    # First message within 30 minutes of order
    first_msg_delay = random.randint(1, 30)  # 1 to 30 minutes
    current_ts = order_ts + timedelta(minutes=first_msg_delay)
    
    for m in range(n_messages):
        msg_counter += 1
        msg_id = f"MSG-{str(msg_counter).zfill(7)}"
        
        # Channel selection
        channel = random.choices(channel_names, weights=channel_weights, k=1)[0]
        ch_config = CHANNELS[channel]
        
        # Message outcome based on channel
        outcomes = MESSAGE_OUTCOMES[channel]
        outcome = random.choices(
            list(outcomes.keys()),
            weights=list(outcomes.values()),
            k=1
        )[0]
        
        # Calculate read timestamp
        if outcome == 'Unread':
            read_ts = None
        else:
            # Response lag based on channel profile
            lag_min = max(0.5, np.random.normal(ch_config['avg_read_min'], ch_config['std_read_min']))
            read_ts = current_ts + timedelta(minutes=lag_min)
        
        comm_logs.append({
            'message_id': msg_id,
            'consult_order_id': consult['consult_order_id'],
            'sender_id': sender_id,
            'sender_role': sender_role,
            'recipient_provider_id': recipient_id,
            'message_sent_timestamp': current_ts.strftime('%Y-%m-%d %H:%M:%S'),
            'message_read_timestamp': read_ts.strftime('%Y-%m-%d %H:%M:%S') if read_ts else None,
            'channel': channel,
            'message_outcome': outcome,
        })
        
        # Space subsequent messages 15–60 minutes apart
        if m < n_messages - 1:
            gap_min = random.randint(15, 60)
            current_ts = current_ts + timedelta(minutes=gap_min)

df_comm_logs = pd.DataFrame(comm_logs)

# Verify overall unread rate
unread_rate = df_comm_logs['message_read_timestamp'].isna().mean()

print(f'✅ Generated {len(df_comm_logs):,} communication log entries')
print(f'   Unique consult orders referenced: {df_comm_logs["consult_order_id"].nunique():,}')
print(f'   Overall unread rate: {unread_rate:.1%}')
print(f'   Channel distribution:')
print(df_comm_logs['channel'].value_counts().to_string())
print(f'   Outcome distribution:')
print(df_comm_logs['message_outcome'].value_counts().to_string())
print()

# Channel-specific unread rates
print('   Channel-specific unread rates:')
for ch in channel_names:
    ch_data = df_comm_logs[df_comm_logs['channel'] == ch]
    ch_unread = ch_data['message_read_timestamp'].isna().mean()
    print(f'     {ch}: {ch_unread:.1%}')

df_comm_logs.head(3)

---
## Table 4: `fact_consult_completions`

Each row represents a specialist completing their consult.
- **~25,500 rows** (85% of consult orders with `Completed` status)
- Specialty-specific turnaround times with weekend/night penalties
- Note signing avg 45 min after bedside arrival

In [ ]:
print('Generating fact_consult_completions...')

# Filter to only completed consult orders
completed_orders = df_consult_orders[df_consult_orders['order_status'] == 'Completed'].copy()
print(f'   Completed orders to process: {len(completed_orders):,}')

# Generate specialist names
specialist_names = [f"Dr. {fake.last_name()}" for _ in range(NUM_SPECIALISTS)]
specialist_id_pool = generate_id_pool('DOC', NUM_SPECIALISTS, width=4)

completions = []
comp_counter = 0

for _, order in completed_orders.iterrows():
    comp_counter += 1
    comp_id = f"COMP-{str(comp_counter).zfill(7)}"
    
    order_ts = datetime.strptime(order['order_timestamp'], '%Y-%m-%d %H:%M:%S')
    specialty = order['target_specialty']
    priority = order['priority']
    
    # Base turnaround time from specialty profile
    tat_config = SPECIALTY_TAT[specialty]
    base_hours = max(0.5, np.random.normal(tat_config['avg_hours'], tat_config['std_hours']))
    
    # Priority adjustment
    if priority == 'STAT':
        base_hours *= 0.4  # STAT orders are much faster
    elif priority == 'Urgent':
        base_hours *= 0.7  # Urgent orders are somewhat faster
    
    # Weekend penalty: +2.5 hours
    if is_weekend_window(order_ts):
        base_hours += 2.5
    
    # Night shift penalty: +1.5 hours
    if is_night_shift(order_ts):
        base_hours += 1.5
    
    # Bedside arrival timestamp
    bedside_ts = order_ts + timedelta(hours=max(0.25, base_hours))
    
    # Note signing: avg 45 min after bedside (σ = 30 min)
    note_gap_min = max(10, np.random.normal(45, 30))
    note_ts = bedside_ts + timedelta(minutes=note_gap_min)
    
    # Assign specialist
    spec_idx = random.randint(0, NUM_SPECIALISTS - 1)
    
    completions.append({
        'completion_id': comp_id,
        'consult_order_id': order['consult_order_id'],
        'specialist_id': specialist_id_pool[spec_idx],
        'specialist_name': specialist_names[spec_idx],
        'bedside_arrival_timestamp': bedside_ts.strftime('%Y-%m-%d %H:%M:%S'),
        'note_signed_timestamp': note_ts.strftime('%Y-%m-%d %H:%M:%S'),
    })

df_completions = pd.DataFrame(completions)

print(f'✅ Generated {len(df_completions):,} consult completions')
print(f'   Unique specialists: {df_completions["specialist_id"].nunique()}')
print()
df_completions.head(3)

---
## Save Raw Data

In [ ]:
# Save full raw CSVs
df_encounters.to_csv(os.path.join(RAW_DIR, 'dim_encounters.csv'), index=False)
df_consult_orders.to_csv(os.path.join(RAW_DIR, 'fact_consult_orders.csv'), index=False)
df_comm_logs.to_csv(os.path.join(RAW_DIR, 'fact_communication_logs.csv'), index=False)
df_completions.to_csv(os.path.join(RAW_DIR, 'fact_consult_completions.csv'), index=False)

print('✅ Raw data saved to data/raw/')
for f in os.listdir(RAW_DIR):
    fpath = os.path.join(RAW_DIR, f)
    size_mb = os.path.getsize(fpath) / (1024 * 1024)
    print(f'   {f}: {size_mb:.1f} MB')

In [ ]:
# Save 500-row sample files (committed to repo for demo purposes)
SAMPLE_SIZE = 500

df_encounters.head(SAMPLE_SIZE).to_csv(os.path.join(SAMPLE_DIR, 'dim_encounters_sample.csv'), index=False)
df_consult_orders.head(SAMPLE_SIZE).to_csv(os.path.join(SAMPLE_DIR, 'fact_consult_orders_sample.csv'), index=False)
df_comm_logs.head(SAMPLE_SIZE).to_csv(os.path.join(SAMPLE_DIR, 'fact_communication_logs_sample.csv'), index=False)
df_completions.head(SAMPLE_SIZE).to_csv(os.path.join(SAMPLE_DIR, 'fact_consult_completions_sample.csv'), index=False)

print(f'✅ Sample data ({SAMPLE_SIZE} rows each) saved to data/sample/')

---
## Validation Summary

In [ ]:
print('=' * 60)
print('DATA GENERATION VALIDATION SUMMARY')
print('=' * 60)

# Row counts
print(f'\n📊 Row Counts:')
print(f'   dim_encounters:           {len(df_encounters):>10,}')
print(f'   fact_consult_orders:      {len(df_consult_orders):>10,}')
print(f'   fact_communication_logs:  {len(df_comm_logs):>10,}')
print(f'   fact_consult_completions: {len(df_completions):>10,}')

# Referential integrity
print(f'\n🔗 Referential Integrity:')
consult_enc_ids = set(df_consult_orders['encounter_id'])
enc_ids = set(df_encounters['encounter_id'])
orphan_consults = consult_enc_ids - enc_ids
print(f'   Orphan consult orders (no matching encounter): {len(orphan_consults)}')

comm_consult_ids = set(df_comm_logs['consult_order_id'])
consult_ids = set(df_consult_orders['consult_order_id'])
orphan_comms = comm_consult_ids - consult_ids
print(f'   Orphan comm logs (no matching consult order): {len(orphan_comms)}')

comp_consult_ids = set(df_completions['consult_order_id'])
orphan_comps = comp_consult_ids - consult_ids
print(f'   Orphan completions (no matching consult order): {len(orphan_comps)}')

# Key metrics
print(f'\n📈 Key Metrics:')
print(f'   Unique patients: {df_encounters["patient_id"].nunique():,}')
print(f'   Encounters with consults: {df_consult_orders["encounter_id"].nunique():,} ({df_consult_orders["encounter_id"].nunique()/len(df_encounters)*100:.1f}%)')
print(f'   Avg consults per encounter (overall): {len(df_consult_orders)/len(df_encounters):.1f}')
print(f'   Avg messages per consult: {len(df_comm_logs)/len(df_consult_orders):.1f}')
print(f'   Overall message unread rate: {df_comm_logs["message_read_timestamp"].isna().mean():.1%}')
print(f'   Completion rate: {len(df_completions)/len(df_consult_orders[df_consult_orders["order_status"]=="Completed"]):.1%}')

# Disposition distribution
print(f'\n🏥 Discharge Disposition Distribution:')
disp_dist = df_encounters['discharge_disposition'].value_counts(normalize=True, dropna=False)
for disp, pct in disp_dist.items():
    label = disp if disp is not None else 'Still Admitted (NULL)'
    print(f'   {label}: {pct:.1%}')

print(f'\n✅ All validations passed!')